# Notebook overview: cleaning and recipe-recognition experiments

This notebook takes the labeled **Video 5 vial-filling images** from initial quality checks through three image-classification experiments. It first checks that the frame labels and recipe references are consistent and that each labeled image can be found and opened. It then trains and evaluates models that try to predict one of **10 recipe IDs (5-14)** from images. The recipe CSV is used to validate labels and explain recipe IDs; recipe details are not model inputs.

## What data was cleaned

- The starting label CSV has **845 frame rows**: **509 active-fill frames** (`in_fill=1`) and **336 between-fill frames** (`in_fill=0`).
- The cleaning cells check duplicate or conflicting filenames, required labels, vial/recipe consistency against the recipe CSV, image availability, and whether the JPEG files can be decoded. Invalid or ambiguous rows are excluded from the cleaned index and described in a QC report.
- Cleaning applies to the **label/index data**, not the original JPEGs. The source CSVs and ZIP archives are left unchanged. The cleaned label table includes validated recipe details from the recipe CSV; the QC report records row-level checks and exclusions.
- The recipe-classification experiments use the **509 active-fill frames**. Frames between fills are retained in the cleaned data but are not recipe-training examples. The model preprocessing masks the timestamp overlay, resizes/pads images, and normalizes them; it does not alter the stored source images.
- The tank index contains image filenames and matching information, **not numeric tank sensor readings**. The tank experiments below use tank-camera images only.

## What was trained

All experiments classify the same 10 recipe IDs using ImageNet-initialized ResNet18 encoders and the same fill-grouped split: **340 training frames/pairs** from two fills per recipe and **169 held-out test frames/pairs** from one separate fill per recipe. Keeping complete fills together reduces the risk that near-duplicate neighboring frames appear in both sets.

1. **Vial-only model:** predicts the recipe from vial-camera images.
2. **Tank-only model:** predicts the recipe from timestamp-matched tank-camera images. It does not use tank measurements.
3. **Vial + tank model:** uses both timestamp-matched images through separate ResNet18 encoders and a shared recipe classifier.

The **Results and interpretation** section compares held-out performance and includes training-versus-test accuracy measured from the saved model checkpoints. Outputs from the cleaning and training steps are saved under `cleaned_data`, including cleaned labels, the image QC report, model checkpoints, split files, predictions, confusion matrices, and metrics.

## How to use this notebook

Read the Markdown before each code cell and run cells from top to bottom. The first section performs the data checks and creates the cleaned label files. The later sections build the split, train each image model, evaluate it, and save its outputs. These are small proof-of-concept experiments: there are only three fills per recipe and all data comes from one recording, so the results should not be treated as a dependable estimate for new production runs. For stronger validation, collect more fills from independent recordings.


# Video 5 image and label cleaning

## What I found and what I recommend

The sampled JPEGs show a fixed-camera vial-filling station: the vial and dispensing nozzle are near the center, while the background is mostly static. The sampled images are 640 x 480 and include a timestamp burned into the lower-left pixels. The label CSV has 845 rows: 509 `in_fill=1` frames and 336 `in_fill=0` (`between_fills`) frames. The `in_fill` and `type` labels agree in the CSV, and all active-fill rows have vial, recipe, and type labels. The ZIP has 1,735 JPEGs; all 845 CSV filenames are present, and the other 890 ZIP images are not referenced by this CSV. They may belong to other footage, so they should not be treated as bad data or pulled into this labeled dataset.

For this dataset, clean the **index and labels** first: normalize whitespace, validate label relationships, identify duplicate/conflicting filenames, check that every labeled JPEG exists and decodes, and validate each active frame's `(vial, recipe_id)` against the vial-recipes CSV. The cleaned output carries recipe details (fill type, amounts, dispense order, and color purpose) from that lookup. Keep `between_fills` as a valid class for fill-state detection; for recipe/type classification, train only on `in_fill=1` rows. This prepares labeled examples for a vision model; it does not train a model or predict labels from new images. Do not delete source images or apply denoising, color correction, or a crop automatically: the sample images look usable, and such transformations could alter the liquid-color cues. Apply resize/normalization as model preprocessing, and consider a task-specific crop only after validating it does not remove useful context. Because neighboring frames are highly similar, split train/validation/test by vial or recording segment rather than randomly by frame to reduce leakage.

This notebook does not overwrite the source CSVs or ZIP. It removes exact duplicate filename rows, flags conflicting duplicate labels for exclusion, validates the label and recipe lookup, and checks that each referenced JPEG can be decoded. Running it creates `cleaned_data/cleaned_video5_frame_labels.csv` (with recipe details) and `cleaned_data/video5_image_qc_report.csv` alongside the inputs. Rows with invalid labels, an unknown vial/recipe pair, or absent, ambiguous, or unreadable images are excluded from the cleaned index; the report records why. The cleaning cells require `pandas` and `Pillow`; the ResNet18 training cell also requires `torch` and `torchvision`.

## Step 1 - Install the packages

This cell installs the libraries used below: pandas reads the CSV files, Pillow opens images, and PyTorch/torchvision provide the ResNet18 model. Run it once per selected notebook environment; restart the kernel if VS Code asks.

In [ ]:
%pip install pandas pillow torch torchvision

## Step 2 - Load the labels and inspect the files

This cell reads the frame-label CSV, vial-recipe CSV, and image ZIP index. It checks required columns, normalizes values, and shows how many rows belong to each fill state and type. It does not edit the original files.

In [ ]:
from collections import defaultdict
from io import BytesIO
from pathlib import Path
import zipfile

import pandas as pd
from PIL import Image, UnidentifiedImageError

LABEL_CSV = Path("video5_frame_labels.csv")
RECIPES_CSV = Path("video5_vial_recipes.csv")
IMAGE_ZIP = Path("images-20261021T001554Z-1-001.zip")
OUTPUT_DIR = Path("cleaned_data")

for input_path in (LABEL_CSV, RECIPES_CSV, IMAGE_ZIP):
    if not input_path.is_file():
        raise FileNotFoundError(
            f"Could not find {input_path}. Run this notebook with the dataset folder as the working directory."
        )

labels = pd.read_csv(LABEL_CSV)
recipes = pd.read_csv(RECIPES_CSV)
required_columns = {
    "filename", "timestamp", "video_s", "in_fill", "vial", "recipe_id", "type",
    "t100_ml", "t200_ml", "t400_ml", "total_ml", "dispense_order", "color_purpose"
}
missing_columns = required_columns.difference(labels.columns)
if missing_columns:
    raise ValueError(f"Label CSV is missing required columns: {sorted(missing_columns)}")
recipe_required_columns = {
    "vial", "recipe_id", "type", "t100_ml", "t200_ml", "t400_ml",
    "total_ml", "dispense_order", "color_purpose"
}
missing_recipe_columns = recipe_required_columns.difference(recipes.columns)
if missing_recipe_columns:
    raise ValueError(
        f"Recipe CSV is missing required columns: {sorted(missing_recipe_columns)}"
    )
for column in ("vial", "recipe_id", "t100_ml", "t200_ml", "t400_ml", "total_ml"):
    recipes[column] = pd.to_numeric(recipes[column], errors="coerce")
if recipes[["vial", "recipe_id"]].isna().any().any():
    raise ValueError("Recipe CSV contains a missing or invalid vial/recipe_id key")
if recipes.duplicated(["vial", "recipe_id"]).any():
    raise ValueError("Recipe CSV has duplicate (vial, recipe_id) keys")
recipe_records = {
    (row["vial"], row["recipe_id"]): row
    for _, row in recipes.iterrows()
}

labels["filename"] = labels["filename"].astype("string").str.strip()
for column in ("timestamp", "type"):
    labels[column] = labels[column].astype("string").str.strip()
for column in ("video_s", "in_fill", "vial", "recipe_id"):
    labels[column] = pd.to_numeric(labels[column], errors="coerce")

with zipfile.ZipFile(IMAGE_ZIP) as image_zip:
    archive_members = defaultdict(list)
    for member in image_zip.namelist():
        if member.lower().endswith((".jpg", ".jpeg")):
            archive_members[Path(member).name].append(member)
    archive_image_count = sum(len(members) for members in archive_members.values())

print(f"CSV rows: {len(labels):,}")
print(f"JPEGs in ZIP: {archive_image_count:,}")
print("Rows by fill state:")
display(labels["in_fill"].value_counts(dropna=False).rename_axis("in_fill").to_frame("rows"))
print("Rows by type:")
display(labels["type"].value_counts(dropna=False).rename_axis("type").to_frame("rows"))

## Step 3 - Check labels and image files

This cell checks for duplicate filenames, invalid or conflicting labels, missing recipe definitions, and images that are missing or cannot be decoded. It creates a row-by-row QC table so any excluded records have a reason.

In [ ]:
duplicate_rows = labels[labels["filename"].duplicated(keep=False)]
conflicting_filenames = set()
for filename, group in duplicate_rows.groupby("filename", dropna=False):
    if len(group.drop_duplicates()) > 1:
        conflicting_filenames.add(filename)

# Identical duplicate records are redundant; conflicting records are retained
# for the report but excluded from the cleaned training index.
labels = labels.drop_duplicates(subset="filename", keep="first").copy()
quality_rows = []

with zipfile.ZipFile(IMAGE_ZIP) as image_zip:
    for row_index, row in labels.iterrows():
        issues = []
        filename = row["filename"]
        if pd.isna(filename) or not filename:
            issues.append("missing_filename")
        if pd.notna(filename) and filename in conflicting_filenames:
            issues.append("conflicting_duplicate_filename")
        if pd.isna(row["video_s"]):
            issues.append("invalid_video_s")
        if row["in_fill"] not in (0, 1):
            issues.append("invalid_in_fill")

        is_active_fill = row["in_fill"] == 1
        if is_active_fill and pd.notna(row["type"]) and row["type"] == "between_fills":
            issues.append("fill_state_type_mismatch")
        elif row["in_fill"] == 0 and (
            pd.isna(row["type"]) or row["type"] != "between_fills"
        ):
            issues.append("fill_state_type_mismatch")
        if is_active_fill and (
            pd.isna(row["vial"]) or pd.isna(row["recipe_id"]) or pd.isna(row["type"])
        ):
            issues.append("missing_active_fill_label")
        elif is_active_fill:
            recipe = recipe_records.get((row["vial"], row["recipe_id"]))
            if recipe is None:
                issues.append("recipe_definition_not_found")
            else:
                recipe_fields = (
                    "type", "t100_ml", "t200_ml", "t400_ml", "total_ml",
                    "dispense_order", "color_purpose"
                )
                for field in recipe_fields:
                    label_value = row[field]
                    recipe_value = recipe[field]
                    both_missing = pd.isna(label_value) and pd.isna(recipe_value)
                    if not both_missing and (
                        pd.isna(label_value)
                        or pd.isna(recipe_value)
                        or label_value != recipe_value
                    ):
                        issues.append(f"recipe_{field}_mismatch")
                        break
        if pd.isna(row["type"]) or not str(row["type"]).strip():
            issues.append("missing_type")

        label_ok = not issues
        candidates = archive_members.get(str(filename), []) if pd.notna(filename) else []
        archive_member = candidates[0] if len(candidates) == 1 else None
        image_ok = False
        if not candidates:
            issues.append("image_not_found_in_zip")
        elif len(candidates) > 1:
            issues.append("ambiguous_image_filename_in_zip")
        else:
            try:
                with image_zip.open(archive_member) as image_file:
                    image_bytes = image_file.read()
                with Image.open(BytesIO(image_bytes)) as image:
                    image.verify()
                image_ok = True
            except (OSError, SyntaxError, UnidentifiedImageError, zipfile.BadZipFile):
                issues.append("image_unreadable_or_corrupt")

        quality_rows.append(
            {
                "row_index": row_index,
                "archive_member": archive_member,
                "image_ok": image_ok,
                "label_ok": label_ok,
                "issues": ";".join(issues),
            }
        )

quality = pd.DataFrame(quality_rows).set_index("row_index")
cleaned = labels.join(quality)
cleaned_labels = cleaned.loc[cleaned["image_ok"] & cleaned["label_ok"]].copy()
recipe_details = recipes.rename(
    columns={
        column: f"recipe_{column}"
        for column in recipes.columns
        if column not in ("vial", "recipe_id")
    }
)
cleaned_labels = cleaned_labels.merge(
    recipe_details,
    on=["vial", "recipe_id"],
    how="left",
    validate="many_to_one",
)
qc_report = cleaned[
    ["filename", "vial", "recipe_id", "type", "archive_member", "image_ok", "label_ok", "issues"]
].copy()

print(f"Duplicate filename rows in input: {len(duplicate_rows):,}")
print(f"Conflicting filenames: {len(conflicting_filenames):,}")
print(f"Rows retained after checks: {len(cleaned_labels):,} / {len(labels):,}")
display(qc_report[qc_report["issues"].ne("")].head(20))

## Step 4 - Save the cleaned labels

This cell writes the cleaned label table and image QC report into `cleaned_data`. It also creates the active-fill-only table used for recipe training. The original CSV and ZIP stay unchanged.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cleaned_labels.to_csv(OUTPUT_DIR / "cleaned_video5_frame_labels.csv", index=False)
qc_report.to_csv(OUTPUT_DIR / "video5_image_qc_report.csv", index=False)

# Keep all valid classes for fill-state detection. Use this subset only when
# the modeling target is recipe or fill type, which is undefined between fills.
recipe_training_labels = cleaned_labels.loc[cleaned_labels["in_fill"].eq(1)].copy()

print(f"Saved cleaned labels and QC report to: {OUTPUT_DIR.resolve()}")
print(f"Fill-state training rows (includes between_fills): {len(cleaned_labels):,}")
print(f"Recipe/type training rows (active fills only): {len(recipe_training_labels):,}")
print("Key recipe fields added: recipe_type, recipe_t100_ml, recipe_t200_ml, recipe_t400_ml, recipe_total_ml, recipe_dispense_order, recipe_color_purpose")
display(
    recipe_training_labels[
        ["filename", "vial", "recipe_id", "type", "recipe_type", "recipe_color_purpose"]
    ].head()
)
display(
    recipe_training_labels.groupby(["recipe_id", "recipe_type"], dropna=False)
    .size()
    .rename("frame_count")
    .reset_index()
)

# Vial-only recipe recognition with ResNet18

This is the image-model experiment. It uses **only vial-camera images** to predict one of 10 specific recipe IDs (5?14). The recipe CSV explains the IDs; those recipe details are not given to the model as inputs. The tank-camera images are not used.

## Step 5 - Choose training and test examples

Only active-fill rows (`in_fill=1`) are used: 509 labeled frames. Each recipe has three separate vial fills. The split keeps one whole vial fill per recipe for testing and uses the other two for training. That avoids putting near-identical frames from the same fill on both sides of the test. The fixed random seed is 42.

## Step 6 - Set paths and training settings

This cell imports the libraries, points to the cleaned labels and image ZIP, and sets the image size, batch size, epoch counts, and reproducibility seed. Run it after the cleaning cells.

In [9]:
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18


ROOT = Path.cwd()
DATA_DIR = ROOT / "cleaned_data"
LABELS_CSV = DATA_DIR / "cleaned_video5_frame_labels.csv"
IMAGE_ZIP = ROOT / "images-20261021T001554Z-1-001.zip"
RECIPES_CSV = ROOT / "video5_vial_recipes.csv"
OUTPUT_DIR = DATA_DIR / "vial_resnet18_run"
SEED = 42
BATCH_SIZE = 32
HEAD_EPOCHS = 4
FINE_TUNE_EPOCHS = 6
IMAGE_SIZE = 224


## Step 7 - Prepare each vial image

The dataset class reads an image from the ZIP, masks the printed timestamp in the lower-left, pads it to 224 - 224, and normalizes colors using ImageNet statistics. Each image is paired with its numeric recipe class. This helps prevent the model from learning the recording timestamp instead of vial appearance.

In [10]:
class VialFrameDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, class_to_index: dict[int, int]) -> None:
        self.rows = rows.reset_index(drop=True)
        self.class_to_index = class_to_index
        self.archive = zipfile.ZipFile(IMAGE_ZIP)
        weights = ResNet18_Weights.DEFAULT
        self.mean = torch.tensor(weights.transforms().mean).view(3, 1, 1)
        self.std = torch.tensor(weights.transforms().std).view(3, 1, 1)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        row = self.rows.iloc[index]
        with self.archive.open(row["archive_member"]) as image_file:
            with Image.open(image_file) as source:
                image = source.convert("RGB")

        width, height = image.size
        mask_height = max(1, round(height * 0.085))
        mask_width = min(width, round(width * 0.58))
        ImageDraw.Draw(image).rectangle(
            (0, height - mask_height, mask_width, height),
            fill=(128, 128, 128),
        )
        image = ImageOps.pad(
            image,
            (IMAGE_SIZE, IMAGE_SIZE),
            method=Image.Resampling.BILINEAR,
            color=(128, 128, 128),
        )
        pixels = np.asarray(image, dtype=np.float32).copy() / 255.0
        tensor = torch.from_numpy(pixels).permute(2, 0, 1)
        tensor = (tensor - self.mean) / self.std
        recipe_id = int(row["recipe_id"])
        return tensor, self.class_to_index[recipe_id]

    def close(self) -> None:
        self.archive.close()


## Step 8 - Define the evaluation measures

This helper compares predictions with the true recipe IDs and calculates accuracy, balanced accuracy, macro F1, per-recipe precision/recall, and a confusion matrix. Balanced accuracy and macro F1 help show whether a model only performs well on some recipes.

In [11]:
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    class_ids: list[int],
    device: torch.device,
) -> tuple[dict[str, object], np.ndarray, list[int], list[int]]:
    model.eval()
    actual: list[int] = []
    predicted: list[int] = []
    with torch.inference_mode():
        for images, labels in loader:
            logits = model(images.to(device))
            predicted.extend(logits.argmax(dim=1).cpu().tolist())
            actual.extend(labels.tolist())

    matrix = np.zeros((len(class_ids), len(class_ids)), dtype=np.int64)
    for truth, guess in zip(actual, predicted):
        matrix[truth, guess] += 1

    per_class: dict[str, dict[str, float | int]] = {}
    recalls: list[float] = []
    f1_scores: list[float] = []
    for index, recipe_id in enumerate(class_ids):
        true_positive = int(matrix[index, index])
        support = int(matrix[index].sum())
        predicted_count = int(matrix[:, index].sum())
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )
        per_class[str(recipe_id)] = {
            "support": support,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        recalls.append(recall)
        f1_scores.append(f1)

    metrics: dict[str, object] = {
        "test_frames": len(actual),
        "accuracy": float(np.mean(np.asarray(actual) == np.asarray(predicted))),
        "balanced_accuracy": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1_scores)),
        "per_recipe": per_class,
    }
    return metrics, matrix, actual, predicted


## Step 9 - Build the leakage-safe split

This cell loads the cleaned table, keeps active-fill rows, chooses one complete vial fill per recipe for the test set, checks that train/test vial IDs do not overlap, and saves the split. It also prints the number of frames and vial fills in each group.

In [12]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

for required in (LABELS_CSV, IMAGE_ZIP, RECIPES_CSV):
    if not required.is_file():
        raise FileNotFoundError(
            f"Required file not found: {required}. Run the cleaning notebook first."
        )

labels = pd.read_csv(LABELS_CSV)
required_columns = {"filename", "archive_member", "in_fill", "vial", "recipe_id"}
missing = required_columns.difference(labels.columns)
if missing:
    raise ValueError(f"Cleaned labels are missing columns: {sorted(missing)}")

active = labels.loc[labels["in_fill"].eq(1)].copy()
if active.empty:
    raise ValueError("No active-fill rows were found in the cleaned labels.")
if active[["archive_member", "vial", "recipe_id"]].isna().any().any():
    raise ValueError("Active-fill rows contain a missing image, vial, or recipe.")

recipe_ids = sorted(active["recipe_id"].astype(int).unique().tolist())
class_to_index = {recipe_id: index for index, recipe_id in enumerate(recipe_ids)}
rng = np.random.default_rng(SEED)
test_vials: set[float] = set()
for recipe_id, recipe_rows in active.groupby("recipe_id"):
    vials = sorted(recipe_rows["vial"].unique().tolist())
    if len(vials) < 3:
        raise ValueError(
            f"Recipe {int(recipe_id)} has only {len(vials)} distinct vial fills; "
            "at least three are needed for this hold-one-vial-out test."
        )
    test_vials.add(float(rng.choice(vials)))

active["split"] = np.where(active["vial"].isin(test_vials), "test", "train")
train_rows = active.loc[active["split"].eq("train")].copy()
test_rows = active.loc[active["split"].eq("test")].copy()
if set(train_rows["vial"]).intersection(test_rows["vial"]):
    raise RuntimeError("Vial leakage detected between train and test.")
train_classes = set(train_rows["recipe_id"].astype(int))
test_classes = set(test_rows["recipe_id"].astype(int))
if train_classes != set(recipe_ids) or test_classes != set(recipe_ids):
    raise RuntimeError("The grouped split does not contain every recipe in both sets.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
active[["filename", "vial", "recipe_id", "split"]].to_csv(
    OUTPUT_DIR / "vial_resnet18_split.csv", index=False
)
split_summary = (
    active.groupby(["split", "recipe_id", "vial"], as_index=False)
    .size()
    .sort_values(["split", "recipe_id", "vial"])
)
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print(f"Active-fill images: {len(active)}")
print(f"Training images: {len(train_rows)}; held-out test images: {len(test_rows)}")
print("Held-out vial per recipe:", sorted(test_vials))
print("Group split (vial counts):")
print(split_summary.groupby(["split", "recipe_id"])["vial"].nunique().unstack().to_string())


Device: CPU
Active-fill images: 509
Training images: 340; held-out test images: 169
Held-out vial per recipe: [1.0, 6.0, 8.0, 11.0, 14.0, 18.0, 19.0, 24.0, 25.0, 28.0]
Group split (vial counts):
recipe_id  5.0   6.0   7.0   8.0   9.0   10.0  11.0  12.0  13.0  14.0
split                                                                
test          1     1     1     1     1     1     1     1     1     1
train         2     2     2     2     2     2     2     2     2     2


## Step 10 - Fine-tune ResNet18

This cell creates image loaders and the 10-class ResNet18. It starts from ImageNet weights, trains the new classification layer for four epochs, then fine-tunes that layer plus ResNet18?s final block (`layer4`) for six more epochs. Class-weighted loss helps account for differences in recipe frame counts. This can take several minutes on CPU.

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = VialFrameDataset(train_rows, class_to_index)
test_dataset = VialFrameDataset(test_rows, class_to_index)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

# Use ImageNet initialization; fail explicitly if the weights cannot be fetched.
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(recipe_ids))
model.to(device)
for parameter in model.parameters():
    parameter.requires_grad = False
for parameter in model.fc.parameters():
    parameter.requires_grad = True

counts = train_rows["recipe_id"].astype(int).value_counts()
class_weights = torch.tensor(
    [
        len(train_rows) / (len(recipe_ids) * int(counts[recipe_id]))
        for recipe_id in recipe_ids
    ],
    dtype=torch.float32,
    device=device,
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
history: list[dict[str, float | int | str]] = []

phases = [
    ("classifier", HEAD_EPOCHS, 1e-3, False),
    ("layer4_finetune", FINE_TUNE_EPOCHS, 1e-4, True),
]
for phase, epochs, learning_rate, unfreeze_layer4 in phases:
    if unfreeze_layer4:
        for parameter in model.layer4.parameters():
            parameter.requires_grad = True
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=0.01,
    )
    for epoch in range(1, epochs + 1):
        model.train()
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        total_loss = 0.0
        seen = 0
        for images, targets in train_loader:
            images = images.to(device)
            targets = targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), targets)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(targets)
            seen += len(targets)
        mean_loss = total_loss / seen
        history.append(
            {
                "phase": phase,
                "epoch": epoch,
                "train_loss": mean_loss,
            }
        )
        print(f"{phase} epoch {epoch}/{epochs}: train_loss={mean_loss:.4f}")


classifier epoch 1/4: train_loss=2.3137
classifier epoch 2/4: train_loss=2.0679
classifier epoch 3/4: train_loss=1.8577
classifier epoch 4/4: train_loss=1.6891
layer4_finetune epoch 1/6: train_loss=1.3116
layer4_finetune epoch 2/6: train_loss=0.4455
layer4_finetune epoch 3/6: train_loss=0.1587
layer4_finetune epoch 4/6: train_loss=0.0912
layer4_finetune epoch 5/6: train_loss=0.0679
layer4_finetune epoch 6/6: train_loss=0.0349


## Step 11 - Test the model and save results

The held-out vial images are evaluated only after training. This cell writes the model weights, per-frame predictions, confusion matrix, split, and metrics into `cleaned_data/vial_resnet18_run`. Running the notebook again retrains the model and replaces the run outputs.

In [14]:
metrics, matrix, actual, predicted = evaluate(
    model, test_loader, recipe_ids, device
)
index_to_class = {index: recipe_id for recipe_id, index in class_to_index.items()}
recipe_metadata = (
    pd.read_csv(RECIPES_CSV)
    .drop_duplicates("recipe_id")
    .set_index("recipe_id")
)
recipe_names = [
    f"{recipe_id}: {recipe_metadata.loc[recipe_id, 'type']}"
    for recipe_id in recipe_ids
]
pd.DataFrame(matrix, index=recipe_names, columns=recipe_names).to_csv(
    OUTPUT_DIR / "vial_resnet18_confusion_matrix.csv"
)
predictions = test_rows.reset_index(drop=True)[
    ["filename", "vial", "recipe_id"]
].copy()
predictions["actual_recipe_id"] = predictions["recipe_id"].astype(int)
predictions["predicted_recipe_id"] = [
    index_to_class[index] for index in predicted
]
predictions["correct"] = (
    predictions["actual_recipe_id"] == predictions["predicted_recipe_id"]
)
predictions.to_csv(OUTPUT_DIR / "vial_resnet18_test_predictions.csv", index=False)
(OUTPUT_DIR / "vial_resnet18_metrics.json").write_text(
    json.dumps(
        {
            **metrics,
            "seed": SEED,
            "test_vials": sorted(test_vials),
            "train_vials": sorted(set(train_rows["vial"].tolist())),
            "recipe_ids": recipe_ids,
            "timestamp_mask": "lower-left 58% width by 8.5% height",
            "initialization": "ImageNet ResNet18 weights",
            "training_history": history,
            "warning": (
                "Small proof-of-concept: three fills per recipe and one held-out "
                "fill per recipe. This is not a robust estimate of deployment "
                "performance; use a separate recording for stronger validation."
            ),
        },
        indent=2,
    ),
    encoding="utf-8",
)
torch.save(
    {
        "model_state_dict": model.cpu().state_dict(),
        "recipe_ids": recipe_ids,
        "class_to_index": class_to_index,
        "image_size": IMAGE_SIZE,
        "seed": SEED,
        "timestamp_mask": "lower-left 58% width by 8.5% height",
    },
    OUTPUT_DIR / "vial_resnet18_recipe_model.pth",
)

print("\nHeld-out test results (one complete vial fill per recipe):")
print(f"Accuracy:          {metrics['accuracy']:.3f}")
print(f"Balanced accuracy: {metrics['balanced_accuracy']:.3f}")
print(f"Macro F1:          {metrics['macro_f1']:.3f}")
print("Per-recipe recall:")
for recipe_id in recipe_ids:
    row = metrics["per_recipe"][str(recipe_id)]
    print(
        f"  Recipe {recipe_id}: recall={row['recall']:.3f}, "
        f"support={row['support']}"
    )
print(f"\nArtifacts saved in: {OUTPUT_DIR}")

train_dataset.close()
test_dataset.close()



Held-out test results (one complete vial fill per recipe):
Accuracy:          0.438
Balanced accuracy: 0.396
Macro F1:          0.342
Per-recipe recall:
  Recipe 5: recall=0.000, support=13
  Recipe 6: recall=0.917, support=12
  Recipe 7: recall=0.083, support=12
  Recipe 8: recall=0.444, support=18
  Recipe 9: recall=0.783, support=23
  Recipe 10: recall=0.316, support=19
  Recipe 11: recall=0.000, support=20
  Recipe 12: recall=1.000, support=22
  Recipe 13: recall=0.421, support=19
  Recipe 14: recall=0.000, support=11

Artifacts saved in: c:\Users\varelaa\Hackathon 2026\Training model 1\cleaned_data\vial_resnet18_run


## Results from the first run

The first run used CPU and ImageNet-pretrained ResNet18. On 169 frames from 10 held-out vial fills, it achieved **43.8% accuracy**, **39.6% balanced accuracy**, and **34.2% macro F1**. Per-recipe recall: recipe 5 0%, 6 91.7%, 7 8.3%, 8 44.4%, 9 78.3%, 10 31.6%, 11 0%, 12 100%, 13 42.1%, and 14 0%. The model is not yet reliable across all recipes. There are only three fills per recipe, so this one-split test is small; validate on more separate fills/recordings before relying on predictions.

## Where the outputs are saved and how to run

Run the notebook with `Training model 1` as the working directory and select the Python kernel containing `pandas`, `Pillow`, `torch`, and `torchvision`. Use **Run All** from the top. The training code is in the notebook cells above, not in a `%run` call. Model weights, metrics, predictions, confusion matrix, and split are in `cleaned_data/vial_resnet18_run`. The source files are not modified.

## Tank-camera experiment - recipe recognition

This model uses only the wide tank-camera images to predict one of 10 specific recipe IDs (5?14). It does not use vial-camera pixels or numeric tank readings. Recipe IDs are the answer labels, not inputs.

In [15]:
from __future__ import annotations

import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18

ROOT = Path.cwd()
DATA_DIR = ROOT / "cleaned_data"
LABELS_CSV = DATA_DIR / "cleaned_video5_frame_labels.csv"
TANK_INDEX_CSV = ROOT / "tank_index.csv"
TANK_IMAGE_ZIP = ROOT / "TanksLabel.zip"
RECIPES_CSV = ROOT / "video5_vial_recipes.csv"
VIAL_SPLIT_CSV = DATA_DIR / "vial_resnet18_run" / "vial_resnet18_split.csv"
TANK_OUTPUT_DIR = DATA_DIR / "tank_resnet18_run"
SEED = 42
BATCH_SIZE = 32
HEAD_EPOCHS = 4
FINE_TUNE_EPOCHS = 6
IMAGE_SIZE = 224



### Step 1 - Match each tank frame to its recipe label

The tank CSV is an image index (filename, timestamp, and video ID), not a sensor-measurement table. This cell pairs tank images to active vial-frame labels using the timestamp-based filenames, requires exact timestamp agreement, filters to video 5, and verifies that each match has one image in the tank ZIP.

In [16]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

for required_file in (LABELS_CSV, TANK_INDEX_CSV, TANK_IMAGE_ZIP, RECIPES_CSV, VIAL_SPLIT_CSV):
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Required input not found: {required_file}. Run the cleaning and vial-model cells first."
        )

tank_labels = pd.read_csv(LABELS_CSV)
tank_labels = tank_labels.loc[tank_labels["in_fill"].eq(1)].copy()
tank_labels["vial_filename"] = tank_labels["filename"]
tank_labels["tank_filename"] = tank_labels["filename"].str.replace(
    "vial_fill_", "tank_level_", regex=False
)
tank_labels = tank_labels.drop(columns="filename")

tank_index = pd.read_csv(TANK_INDEX_CSV)
required_index_columns = {"filename", "timestamp", "video_id"}
missing_index_columns = required_index_columns.difference(tank_index.columns)
if missing_index_columns:
    raise ValueError(f"Tank index is missing columns: {sorted(missing_index_columns)}")
tank_index = tank_index.loc[tank_index["video_id"].eq(5)].copy()
if tank_index["filename"].duplicated().any():
    raise ValueError("Tank index contains duplicate filenames for video 5.")

tank_frames = tank_labels.merge(
    tank_index,
    left_on="tank_filename",
    right_on="filename",
    how="left",
    validate="one_to_one",
    suffixes=("_vial", "_tank"),
    indicator=True,
)
if not tank_frames["_merge"].eq("both").all():
    missing_frames = tank_frames.loc[
        tank_frames["_merge"].ne("both"), "tank_filename"
    ].head(10).tolist()
    raise ValueError(f"Tank images were not indexed for these vial frames: {missing_frames}")
if not tank_frames["timestamp_vial"].eq(tank_frames["timestamp_tank"]).all():
    raise ValueError("Matched vial and tank timestamps do not agree exactly.")
tank_frames = tank_frames.drop(columns="_merge")
tank_frames = tank_frames.rename(columns={"filename": "tank_index_filename"})

with zipfile.ZipFile(TANK_IMAGE_ZIP) as tank_archive:
    tank_members_by_name: dict[str, list[str]] = {}
    for member in tank_archive.namelist():
        if member.lower().endswith((".jpg", ".jpeg")):
            tank_members_by_name.setdefault(Path(member).name, []).append(member)

def find_tank_member(filename: str) -> str:
    matches = tank_members_by_name.get(filename, [])
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one tank image named {filename!r}; found {len(matches)}."
        )
    return matches[0]

tank_frames["tank_archive_member"] = tank_frames["tank_filename"].map(find_tank_member)



### Step 2 - Reuse the vial-model train/test split

To compare tank and vial models fairly, this cell reuses the exact vial-grouped split: 340 training frames and 169 test frames. One whole vial fill per recipe is held out; no frames from that fill can appear in training.

In [17]:
vial_split = pd.read_csv(VIAL_SPLIT_CSV)
tank_frames = tank_frames.merge(
    vial_split[["filename", "vial", "recipe_id", "split"]],
    left_on=["vial_filename", "vial", "recipe_id"],
    right_on=["filename", "vial", "recipe_id"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_split"),
)
if tank_frames["split"].isna().any():
    raise ValueError("Some tank frames are missing from the existing vial-model split.")

tank_train_rows = tank_frames.loc[tank_frames["split"].eq("train")].copy()
tank_test_rows = tank_frames.loc[tank_frames["split"].eq("test")].copy()
tank_recipe_ids = sorted(tank_frames["recipe_id"].astype(int).unique().tolist())
tank_class_to_index = {
    recipe_id: index for index, recipe_id in enumerate(tank_recipe_ids)
}
if len(tank_frames) != 509 or len(tank_train_rows) != 340 or len(tank_test_rows) != 169:
    raise ValueError(
        "Unexpected paired tank-image counts; expected 509 total, 340 train, and 169 test."
    )
if set(tank_train_rows["vial"]).intersection(tank_test_rows["vial"]):
    raise RuntimeError("Vial leakage detected in the shared tank train/test split.")

TANK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tank_frames[["filename", "tank_filename", "vial", "recipe_id", "split"]].to_csv(
    TANK_OUTPUT_DIR / "tank_resnet18_split.csv", index=False
)

print(f"Active vial frames with exact tank-image matches: {len(tank_frames)}")
print(f"Training tank images: {len(tank_train_rows)}")
print(f"Held-out tank images: {len(tank_test_rows)}")
print("Reusing the vial-model split: one complete held-out vial per recipe.")
print("Timestamp pairs match exactly; all matched tank images are from video 5.")



Active vial frames with exact tank-image matches: 509
Training tank images: 340
Held-out tank images: 169
Reusing the vial-model split: one complete held-out vial per recipe.
Timestamp pairs match exactly; all matched tank images are from video 5.


### Step 3 - Prepare tank images

This cell defines how images are read from the tank ZIP. It masks the printed timestamp in the lower-left corner, pads each frame to 224 - 224, normalizes it with ImageNet statistics, and pairs it with the recipe label.

In [18]:
class TankFrameDataset(Dataset):
    def __init__(
        self, rows: pd.DataFrame, class_to_index: dict[int, int]
    ) -> None:
        self.rows = rows.reset_index(drop=True)
        self.class_to_index = class_to_index
        self.archive = zipfile.ZipFile(TANK_IMAGE_ZIP)
        weights = ResNet18_Weights.DEFAULT
        self.mean = torch.tensor(weights.transforms().mean).view(3, 1, 1)
        self.std = torch.tensor(weights.transforms().std).view(3, 1, 1)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        row = self.rows.iloc[index]
        with self.archive.open(row["tank_archive_member"]) as image_file:
            with Image.open(image_file) as source:
                image = source.convert("RGB")

        width, height = image.size
        mask_height = max(1, round(height * 0.085))
        mask_width = min(width, round(width * 0.58))
        ImageDraw.Draw(image).rectangle(
            (0, height - mask_height, mask_width, height),
            fill=(128, 128, 128),
        )
        image = ImageOps.pad(
            image,
            (IMAGE_SIZE, IMAGE_SIZE),
            method=Image.Resampling.BILINEAR,
            color=(128, 128, 128),
        )
        pixels = np.asarray(image, dtype=np.float32).copy() / 255.0
        tensor = torch.from_numpy(pixels).permute(2, 0, 1)
        tensor = (tensor - self.mean) / self.std
        recipe_id = int(row["recipe_id"])
        return tensor, self.class_to_index[recipe_id]

    def close(self) -> None:
        self.archive.close()



### Step 4 - Define how predictions are scored

This helper compares predicted and actual recipe IDs on the held-out fills. It calculates overall accuracy, balanced accuracy, macro F1, per-recipe results, and a confusion matrix.

In [19]:
def evaluate_tank_model(
    model: nn.Module,
    loader: DataLoader,
    recipe_ids: list[int],
    device: torch.device,
) -> tuple[dict[str, object], np.ndarray, list[int], list[int]]:
    model.eval()
    actual: list[int] = []
    predicted: list[int] = []
    with torch.inference_mode():
        for images, labels in loader:
            logits = model(images.to(device))
            predicted.extend(logits.argmax(dim=1).cpu().tolist())
            actual.extend(labels.tolist())

    matrix = np.zeros((len(recipe_ids), len(recipe_ids)), dtype=np.int64)
    for truth, guess in zip(actual, predicted):
        matrix[truth, guess] += 1

    per_recipe: dict[str, dict[str, float | int]] = {}
    recalls: list[float] = []
    f1_scores: list[float] = []
    for index, recipe_id in enumerate(recipe_ids):
        true_positive = int(matrix[index, index])
        support = int(matrix[index].sum())
        predicted_count = int(matrix[:, index].sum())
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )
        per_recipe[str(recipe_id)] = {
            "support": support,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        recalls.append(recall)
        f1_scores.append(f1)

    metrics: dict[str, object] = {
        "test_frames": len(actual),
        "accuracy": float(np.mean(np.asarray(actual) == np.asarray(predicted))),
        "balanced_accuracy": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1_scores)),
        "per_recipe": per_recipe,
    }
    return metrics, matrix, actual, predicted



### Step 5 - Train the tank-only ResNet18

The model starts with ImageNet-pretrained ResNet18 weights and replaces the final layer with 10 recipe classes. It trains the new classifier for 4 epochs, then fine-tunes the classifier and final ResNet block for 6 epochs. It uses class-weighted loss and the same settings and random seed as the vial-only run. On this machine it uses CPU, so training takes a few minutes.

In [20]:
tank_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tank_train_dataset = TankFrameDataset(tank_train_rows, tank_class_to_index)
tank_test_dataset = TankFrameDataset(tank_test_rows, tank_class_to_index)
tank_train_loader = DataLoader(
    tank_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
tank_test_loader = DataLoader(
    tank_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

# Start from the same ImageNet-pretrained ResNet18 weights as the vial-only model.
tank_model = resnet18(weights=ResNet18_Weights.DEFAULT)
tank_model.fc = nn.Linear(tank_model.fc.in_features, len(tank_recipe_ids))
tank_model.to(tank_device)
for parameter in tank_model.parameters():
    parameter.requires_grad = False
for parameter in tank_model.fc.parameters():
    parameter.requires_grad = True

tank_class_counts = tank_train_rows["recipe_id"].astype(int).value_counts()
tank_class_weights = torch.tensor(
    [
        len(tank_train_rows)
        / (len(tank_recipe_ids) * int(tank_class_counts[recipe_id]))
        for recipe_id in tank_recipe_ids
    ],
    dtype=torch.float32,
    device=tank_device,
)
tank_criterion = nn.CrossEntropyLoss(weight=tank_class_weights)
tank_history: list[dict[str, float | int | str]] = []

for phase, epochs, learning_rate, fine_tune_layer4 in [
    ("classifier", HEAD_EPOCHS, 1e-3, False),
    ("layer4_finetune", FINE_TUNE_EPOCHS, 1e-4, True),
]:
    if fine_tune_layer4:
        for parameter in tank_model.layer4.parameters():
            parameter.requires_grad = True
    optimizer = torch.optim.AdamW(
        [
            parameter
            for parameter in tank_model.parameters()
            if parameter.requires_grad
        ],
        lr=learning_rate,
        weight_decay=0.01,
    )
    for epoch in range(1, epochs + 1):
        tank_model.train()
        for module in tank_model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        total_loss = 0.0
        seen = 0
        for images, targets in tank_train_loader:
            images = images.to(tank_device)
            targets = targets.to(tank_device)
            optimizer.zero_grad(set_to_none=True)
            loss = tank_criterion(tank_model(images), targets)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(targets)
            seen += len(targets)
        mean_loss = total_loss / seen
        tank_history.append(
            {"phase": phase, "epoch": epoch, "train_loss": mean_loss}
        )
        print(f"{phase} epoch {epoch}/{epochs}: train_loss={mean_loss:.4f}")



classifier epoch 1/4: train_loss=2.3764
classifier epoch 2/4: train_loss=2.3404
classifier epoch 3/4: train_loss=2.2944
classifier epoch 4/4: train_loss=2.2710
layer4_finetune epoch 1/6: train_loss=2.3781
layer4_finetune epoch 2/6: train_loss=2.2660
layer4_finetune epoch 3/6: train_loss=2.1750
layer4_finetune epoch 4/6: train_loss=1.9764
layer4_finetune epoch 5/6: train_loss=1.7109
layer4_finetune epoch 6/6: train_loss=1.5197


### Step 6 - Evaluate and save the result

After training, this cell evaluates only the held-out tank images, saves the model, predictions, confusion matrix, split, and metrics to `cleaned_data/tank_resnet18_run`, and prints the scores. The test set is small and comes from one recording, so treat it as an initial comparison rather than a reliable estimate of future performance.

In [21]:
tank_metrics, tank_matrix, tank_actual, tank_predicted = evaluate_tank_model(
    tank_model, tank_test_loader, tank_recipe_ids, tank_device
)
tank_index_to_class = {
    index: recipe_id for recipe_id, index in tank_class_to_index.items()
}
tank_recipe_metadata = (
    pd.read_csv(RECIPES_CSV).drop_duplicates("recipe_id").set_index("recipe_id")
)
tank_recipe_names = [
    f"{recipe_id}: {tank_recipe_metadata.loc[recipe_id, 'type']}"
    for recipe_id in tank_recipe_ids
]
pd.DataFrame(
    tank_matrix, index=tank_recipe_names, columns=tank_recipe_names
).to_csv(TANK_OUTPUT_DIR / "tank_resnet18_confusion_matrix.csv")

tank_predictions = tank_test_rows.reset_index(drop=True)[
    ["filename", "tank_filename", "vial", "recipe_id"]
].copy()
tank_predictions["actual_recipe_id"] = tank_predictions["recipe_id"].astype(int)
tank_predictions["predicted_recipe_id"] = [
    tank_index_to_class[index] for index in tank_predicted
]
tank_predictions["correct"] = (
    tank_predictions["actual_recipe_id"]
    == tank_predictions["predicted_recipe_id"]
)
tank_predictions.to_csv(
    TANK_OUTPUT_DIR / "tank_resnet18_test_predictions.csv", index=False
)

tank_metrics["seed"] = SEED
tank_metrics["test_vials"] = sorted(tank_test_rows["vial"].unique().tolist())
tank_metrics["train_vials"] = sorted(tank_train_rows["vial"].unique().tolist())
tank_metrics["recipe_ids"] = tank_recipe_ids
tank_metrics["training_history"] = tank_history
tank_metrics["image_source"] = "TanksLabel.zip; tank_index.csv; video_id=5 only"
tank_metrics["split_source"] = "same vial-grouped split as vial_resnet18_split.csv"
tank_metrics["timestamp_mask"] = "lower-left 58% width by 8.5% height"
tank_metrics["warning"] = (
    "Small proof-of-concept: three fills per recipe and one held-out fill per "
    "recipe. The result is not a robust estimate of performance on new recordings."
)
(TANK_OUTPUT_DIR / "tank_resnet18_metrics.json").write_text(
    json.dumps(tank_metrics, indent=2), encoding="utf-8"
)
torch.save(
    {
        "model_state_dict": tank_model.cpu().state_dict(),
        "recipe_ids": tank_recipe_ids,
        "class_to_index": tank_class_to_index,
        "image_size": IMAGE_SIZE,
        "seed": SEED,
        "timestamp_mask": "lower-left 58% width by 8.5% height",
        "input": "tank camera images only",
    },
    TANK_OUTPUT_DIR / "tank_resnet18_recipe_model.pth",
)

print("\nTank-only held-out test results (same held-out vials as vial model):")
print(f"Accuracy:          {tank_metrics['accuracy']:.3f}")
print(f"Balanced accuracy: {tank_metrics['balanced_accuracy']:.3f}")
print(f"Macro F1:          {tank_metrics['macro_f1']:.3f}")
print("Per-recipe recall:")
for recipe_id in tank_recipe_ids:
    result = tank_metrics["per_recipe"][str(recipe_id)]
    print(
        f"  Recipe {recipe_id}: recall={result['recall']:.3f}, "
        f"support={result['support']}"
    )
print(f"\nTank-only model artifacts saved in: {TANK_OUTPUT_DIR}")

tank_train_dataset.close()
tank_test_dataset.close()



Tank-only held-out test results (same held-out vials as vial model):
Accuracy:          0.272
Balanced accuracy: 0.273
Macro F1:          0.162
Per-recipe recall:
  Recipe 5: recall=0.000, support=13
  Recipe 6: recall=1.000, support=12
  Recipe 7: recall=0.000, support=12
  Recipe 8: recall=0.000, support=18
  Recipe 9: recall=0.043, support=23
  Recipe 10: recall=0.105, support=19
  Recipe 11: recall=1.000, support=20
  Recipe 12: recall=0.000, support=22
  Recipe 13: recall=0.579, support=19
  Recipe 14: recall=0.000, support=11

Tank-only model artifacts saved in: c:\Users\varelaa\Hackathon 2026\Training model 1\cleaned_data\tank_resnet18_run


## Tank-only results

The model was trained on 340 tank images and evaluated on 169 held-out images, using the same vial-grouped split as the vial-only model. It achieved **27.2% accuracy**, **27.3% balanced accuracy**, and **16.2% macro F1**. Per-recipe recall: recipe 5: 0.0%, recipe 6: 100.0%, recipe 7: 0.0%, recipe 8: 0.0%, recipe 9: 4.3%, recipe 10: 10.5%, recipe 11: 100.0%, recipe 12: 0.0%, recipe 13: 57.9%, recipe 14: 0.0%.

On this one small held-out split, the tank-only model performed below the vial-only model (43.8% accuracy, 39.6% balanced accuracy, 34.2% macro F1). The result suggests these tank images alone are not as informative for this recipe task in the current dataset. It is not a final conclusion: there are only three fills per recipe, and all images come from one recording. Use a separate recording and more fill cycles for a stronger evaluation.

# Vial + tank recipe recognition

This final experiment uses both cameras together to predict one of the 10 specific recipe IDs. It pairs each vial frame with the tank frame captured at the same timestamp. Neither CSV recipe details nor numeric tank measurements are fed to the model; the recipe ID is only the answer label. This is a two-stream ResNet18 image model: one image encoder per camera, followed by a shared classifier.

## Step 1 - Set up the experiment

This cell imports the libraries, names the source files and output folder, and sets the same random seed and training settings used in the single-camera experiments. It writes results separately, so vial-only and tank-only artifacts are preserved.

In [22]:
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18

ROOT = Path.cwd()
DATA_DIR = ROOT / "cleaned_data"
LABELS_CSV = DATA_DIR / "cleaned_video5_frame_labels.csv"
VIAL_IMAGE_ZIP = ROOT / "images-20261021T001554Z-1-001.zip"
TANK_INDEX_CSV = ROOT / "tank_index.csv"
TANK_IMAGE_ZIP = ROOT / "TanksLabel.zip"
VIAL_SPLIT_CSV = DATA_DIR / "vial_resnet18_run" / "vial_resnet18_split.csv"
RECIPE_CSV = ROOT / "video5_vial_recipes.csv"
FUSION_OUTPUT_DIR = DATA_DIR / "vial_tank_resnet18_run"
SEED = 42
BATCH_SIZE = 16
HEAD_EPOCHS = 4
FINE_TUNE_EPOCHS = 6
IMAGE_SIZE = 224



## Step 2 - Match the two camera views

The code keeps active-fill frames and matches the vial image to its tank image using the timestamp-based filename. It verifies that the tank index says video 5, timestamps match exactly, and both images exist uniquely in their ZIP archives.

In [23]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

for required_file in (
    LABELS_CSV,
    VIAL_IMAGE_ZIP,
    TANK_INDEX_CSV,
    TANK_IMAGE_ZIP,
    VIAL_SPLIT_CSV,
    RECIPE_CSV,
):
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Required input not found: {required_file}. Run the cleaning and "
            "single-camera model cells first."
        )

fusion_rows = pd.read_csv(LABELS_CSV)
fusion_rows = fusion_rows.loc[fusion_rows["in_fill"].eq(1)].copy()
fusion_rows["vial_filename"] = fusion_rows["filename"]
fusion_rows["tank_filename"] = fusion_rows["filename"].str.replace(
    "vial_fill_", "tank_level_", regex=False
)
fusion_rows = fusion_rows.drop(columns=["filename"])

tank_index = pd.read_csv(TANK_INDEX_CSV)
tank_index = tank_index.loc[tank_index["video_id"].eq(5)].copy()
if tank_index["filename"].duplicated().any():
    raise ValueError("Tank index contains duplicate filenames for video 5.")

fusion_rows = fusion_rows.merge(
    tank_index[["filename", "timestamp"]],
    left_on="tank_filename",
    right_on="filename",
    how="left",
    validate="one_to_one",
    suffixes=("_vial", "_tank"),
    indicator=True,
)
if not fusion_rows["_merge"].eq("both").all():
    missing = fusion_rows.loc[
        fusion_rows["_merge"].ne("both"), "tank_filename"
    ].head(10).tolist()
    raise ValueError(f"Could not match these tank frames: {missing}")
if not fusion_rows["timestamp_vial"].eq(fusion_rows["timestamp_tank"]).all():
    raise ValueError("Matched vial and tank frame timestamps do not agree.")
fusion_rows = fusion_rows.drop(columns=["_merge", "filename"])

def unique_archive_member(archive: zipfile.ZipFile, filename: str) -> str:
    matches = [
        member
        for member in archive.namelist()
        if Path(member).name == filename
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one archive image named {filename!r}; "
            f"found {len(matches)}."
        )
    return matches[0]

with zipfile.ZipFile(VIAL_IMAGE_ZIP) as vial_archive:
    fusion_rows["vial_archive_member"] = fusion_rows["vial_filename"].map(
        lambda filename: unique_archive_member(vial_archive, filename)
    )
with zipfile.ZipFile(TANK_IMAGE_ZIP) as tank_archive:
    fusion_rows["tank_archive_member"] = fusion_rows["tank_filename"].map(
        lambda filename: unique_archive_member(tank_archive, filename)
    )



## Step 3 - Reuse the same held-out fills

The multimodal model uses exactly the split from the vial-only model: 340 training pairs and 169 test pairs. One whole vial fill per recipe remains held out. Reusing this split makes the three experiments comparable and prevents frames from the same fill leaking across train and test.

In [24]:
vial_split = pd.read_csv(VIAL_SPLIT_CSV)
fusion_rows = fusion_rows.merge(
    vial_split[["filename", "vial", "recipe_id", "split"]],
    left_on=["vial_filename", "vial", "recipe_id"],
    right_on=["filename", "vial", "recipe_id"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_split"),
)
if fusion_rows["split"].isna().any():
    raise ValueError("Some paired images are missing from the established split.")
fusion_rows = fusion_rows.drop(columns=["filename"])

recipe_ids = sorted(fusion_rows["recipe_id"].astype(int).unique().tolist())
class_to_index = {recipe_id: index for index, recipe_id in enumerate(recipe_ids)}
train_rows = fusion_rows.loc[fusion_rows["split"].eq("train")].copy()
test_rows = fusion_rows.loc[fusion_rows["split"].eq("test")].copy()
if len(fusion_rows) != 509 or len(train_rows) != 340 or len(test_rows) != 169:
    raise ValueError(
        "Expected 509 paired images: 340 for training and 169 for testing."
    )
if set(train_rows["vial"]).intersection(test_rows["vial"]):
    raise RuntimeError("Vial leakage detected between fusion train and test sets.")

FUSION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fusion_rows[
    ["vial_filename", "tank_filename", "vial", "recipe_id", "split"]
].to_csv(FUSION_OUTPUT_DIR / "fusion_resnet18_split.csv", index=False)
print(f"Matched image pairs: {len(fusion_rows)}")
print(f"Training pairs: {len(train_rows)}; held-out test pairs: {len(test_rows)}")
print("The test set reuses the exact vial-only and tank-only split.")



Matched image pairs: 509
Training pairs: 340; held-out test pairs: 169
The test set reuses the exact vial-only and tank-only split.


## Step 4 - Load and prepare paired images

For each example, this cell reads one vial image and its timestamp-matched tank image. It masks the printed timestamp in the lower-left of each frame, pads both images to 224 x 224, and applies ImageNet color normalization. The dataset returns two image tensors and one recipe label.

In [25]:
class PairedCameraDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, labels: dict[int, int]) -> None:
        self.rows = rows.reset_index(drop=True)
        self.class_to_index = labels
        self.vial_archive = zipfile.ZipFile(VIAL_IMAGE_ZIP)
        self.tank_archive = zipfile.ZipFile(TANK_IMAGE_ZIP)
        weights = ResNet18_Weights.DEFAULT
        self.mean = torch.tensor(weights.transforms().mean).view(3, 1, 1)
        self.std = torch.tensor(weights.transforms().std).view(3, 1, 1)

    def __len__(self) -> int:
        return len(self.rows)

    def _read_image(self, archive: zipfile.ZipFile, member: str) -> torch.Tensor:
        with archive.open(member) as image_file:
            with Image.open(image_file) as source:
                image = source.convert("RGB")

        width, height = image.size
        mask_height = max(1, round(height * 0.085))
        mask_width = min(width, round(width * 0.58))
        ImageDraw.Draw(image).rectangle(
            (0, height - mask_height, mask_width, height),
            fill=(128, 128, 128),
        )
        image = ImageOps.pad(
            image,
            (IMAGE_SIZE, IMAGE_SIZE),
            method=Image.Resampling.BILINEAR,
            color=(128, 128, 128),
        )
        pixels = np.asarray(image, dtype=np.float32).copy() / 255.0
        tensor = torch.from_numpy(pixels).permute(2, 0, 1)
        return (tensor - self.mean) / self.std

    def __getitem__(
        self, index: int
    ) -> tuple[torch.Tensor, torch.Tensor, int]:
        row = self.rows.iloc[index]
        vial_image = self._read_image(
            self.vial_archive, row["vial_archive_member"]
        )
        tank_image = self._read_image(
            self.tank_archive, row["tank_archive_member"]
        )
        recipe_id = int(row["recipe_id"])
        return vial_image, tank_image, self.class_to_index[recipe_id]

    def close(self) -> None:
        self.vial_archive.close()
        self.tank_archive.close()



## Step 5 - Build the two-camera model and test measures

Each image goes through its own ImageNet-pretrained ResNet18. The two feature vectors are concatenated and sent to a 10-class recipe classifier. The evaluation helper calculates accuracy, balanced accuracy, macro F1, per-recipe scores, and the confusion matrix.

In [26]:
class TwoCameraResNet18(nn.Module):
    def __init__(self, classes: int) -> None:
        super().__init__()
        self.vial_encoder = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.tank_encoder = resnet18(weights=ResNet18_Weights.DEFAULT)
        vial_features = self.vial_encoder.fc.in_features
        tank_features = self.tank_encoder.fc.in_features
        self.vial_encoder.fc = nn.Identity()
        self.tank_encoder.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.25),
            nn.Linear(vial_features + tank_features, classes),
        )

    def forward(
        self, vial_images: torch.Tensor, tank_images: torch.Tensor
    ) -> torch.Tensor:
        vial_features = self.vial_encoder(vial_images)
        tank_features = self.tank_encoder(tank_images)
        return self.classifier(
            torch.cat((vial_features, tank_features), dim=1)
        )


def evaluate_fusion_model(
    model: nn.Module,
    loader: DataLoader,
    classes: int,
    device: torch.device,
) -> tuple[dict[str, object], np.ndarray, list[int], list[int]]:
    model.eval()
    actual: list[int] = []
    predicted: list[int] = []
    with torch.inference_mode():
        for vial_images, tank_images, labels in loader:
            logits = model(vial_images.to(device), tank_images.to(device))
            predicted.extend(logits.argmax(dim=1).cpu().tolist())
            actual.extend(labels.tolist())

    matrix = np.zeros((classes, classes), dtype=np.int64)
    for truth, guess in zip(actual, predicted):
        matrix[truth, guess] += 1

    per_recipe: dict[str, dict[str, float | int]] = {}
    recalls: list[float] = []
    f1_scores: list[float] = []
    for index in range(classes):
        true_positive = int(matrix[index, index])
        support = int(matrix[index].sum())
        predicted_count = int(matrix[:, index].sum())
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )
        per_recipe[str(recipe_ids[index])] = {
            "support": support,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        recalls.append(recall)
        f1_scores.append(f1)

    metrics: dict[str, object] = {
        "test_frames": len(actual),
        "accuracy": float(np.mean(np.asarray(actual) == np.asarray(predicted))),
        "balanced_accuracy": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1_scores)),
        "per_recipe": per_recipe,
    }
    return metrics, matrix, actual, predicted



## Step 6 - Train the two-stream ResNet18

The new classifier is trained for 4 epochs while the image encoders are frozen. Then the classifier and final ResNet block in each encoder are fine-tuned for 6 epochs. Training uses class-weighted cross entropy and AdamW. The notebook selects CUDA if available; otherwise, it trains on CPU. On this computer the available build is CPU-only, so expect several minutes.

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = PairedCameraDataset(train_rows, class_to_index)
test_dataset = PairedCameraDataset(test_rows, class_to_index)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

model = TwoCameraResNet18(classes=len(recipe_ids)).to(device)
for parameter in model.parameters():
    parameter.requires_grad = False
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

class_counts = train_rows["recipe_id"].astype(int).value_counts()
class_weights = torch.tensor(
    [
        len(train_rows) / (len(recipe_ids) * int(class_counts[recipe_id]))
        for recipe_id in recipe_ids
    ],
    dtype=torch.float32,
    device=device,
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
training_history: list[dict[str, float | int | str]] = []

for phase, epochs, learning_rate, fine_tune_encoders in [
    ("fusion_head", HEAD_EPOCHS, 1e-3, False),
    ("final_blocks", FINE_TUNE_EPOCHS, 1e-4, True),
]:
    if fine_tune_encoders:
        for encoder in (model.vial_encoder, model.tank_encoder):
            for parameter in encoder.layer4.parameters():
                parameter.requires_grad = True
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=0.01,
    )
    for epoch in range(1, epochs + 1):
        model.train()
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        total_loss = 0.0
        seen = 0
        for vial_images, tank_images, targets in train_loader:
            vial_images = vial_images.to(device)
            tank_images = tank_images.to(device)
            targets = targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(vial_images, tank_images), targets)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(targets)
            seen += len(targets)
        mean_loss = total_loss / seen
        training_history.append(
            {"phase": phase, "epoch": epoch, "train_loss": mean_loss}
        )
        print(f"{phase} epoch {epoch}/{epochs}: train_loss={mean_loss:.4f}")



fusion_head epoch 1/4: train_loss=2.4004
fusion_head epoch 2/4: train_loss=2.0503
fusion_head epoch 3/4: train_loss=1.8516
fusion_head epoch 4/4: train_loss=1.6348
final_blocks epoch 1/6: train_loss=1.1760
final_blocks epoch 2/6: train_loss=0.3406
final_blocks epoch 3/6: train_loss=0.1296
final_blocks epoch 4/6: train_loss=0.0975
final_blocks epoch 5/6: train_loss=0.0654
final_blocks epoch 6/6: train_loss=0.1972


## Step 7 - Evaluate and save the multimodal model

Only the held-out image pairs are evaluated after training. This cell saves the checkpoint, per-frame predictions, confusion matrix, metrics, and exact split into `cleaned_data/vial_tank_resnet18_run`. Compare its scores with both single-camera models. Since there are only three fills per recipe and all views come from one recording, treat the scores as an initial experiment, not a reliable estimate for new recordings.

In [28]:
metrics, confusion, actual, predicted = evaluate_fusion_model(
    model, test_loader, len(recipe_ids), device
)
index_to_recipe = {index: recipe for recipe, index in class_to_index.items()}
recipes = pd.read_csv(RECIPE_CSV).drop_duplicates("recipe_id").set_index("recipe_id")
recipe_names = [
    f"{recipe_id}: {recipes.loc[recipe_id, 'type']}" for recipe_id in recipe_ids
]
pd.DataFrame(confusion, index=recipe_names, columns=recipe_names).to_csv(
    FUSION_OUTPUT_DIR / "fusion_resnet18_confusion_matrix.csv"
)

predictions = test_rows.reset_index(drop=True)[
    ["vial_filename", "tank_filename", "vial", "recipe_id"]
].copy()
predictions["actual_recipe_id"] = predictions["recipe_id"].astype(int)
predictions["predicted_recipe_id"] = [
    index_to_recipe[index] for index in predicted
]
predictions["correct"] = (
    predictions["actual_recipe_id"] == predictions["predicted_recipe_id"]
)
predictions.to_csv(
    FUSION_OUTPUT_DIR / "fusion_resnet18_test_predictions.csv", index=False
)

metrics.update(
    {
        "seed": SEED,
        "test_vials": sorted(test_rows["vial"].unique().tolist()),
        "train_vials": sorted(train_rows["vial"].unique().tolist()),
        "recipe_ids": recipe_ids,
        "architecture": "two ResNet18 encoders; concatenated features; 10-class head",
        "inputs": "timestamp-masked vial and matched tank images",
        "split_source": "vial_resnet18_split.csv (identical for all modalities)",
        "timestamp_mask": "lower-left 58% width by 8.5% height",
        "training_history": training_history,
        "warning": (
            "Small proof-of-concept: three fills per recipe and one held-out fill "
            "per recipe, all from one recording. The result is not a robust "
            "estimate of performance on new recordings."
        ),
    }
)
(FUSION_OUTPUT_DIR / "fusion_resnet18_metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)
torch.save(
    {
        "model_state_dict": model.cpu().state_dict(),
        "recipe_ids": recipe_ids,
        "class_to_index": class_to_index,
        "image_size": IMAGE_SIZE,
        "seed": SEED,
        "architecture": "two ResNet18 encoders; concatenate features",
        "timestamp_mask": "lower-left 58% width by 8.5% height",
    },
    FUSION_OUTPUT_DIR / "fusion_resnet18_recipe_model.pth",
)

print("\nTwo-camera held-out test results (same held-out vials as single-camera runs):")
print(f"Accuracy:          {metrics['accuracy']:.3f}")
print(f"Balanced accuracy: {metrics['balanced_accuracy']:.3f}")
print(f"Macro F1:          {metrics['macro_f1']:.3f}")
print("Per-recipe recall:")
for recipe_id in recipe_ids:
    result = metrics["per_recipe"][str(recipe_id)]
    print(
        f"  Recipe {recipe_id}: recall={result['recall']:.3f}, "
        f"support={result['support']}"
    )
print(f"\nTwo-camera artifacts saved in: {FUSION_OUTPUT_DIR}")

train_dataset.close()
test_dataset.close()



Two-camera held-out test results (same held-out vials as single-camera runs):
Accuracy:          0.391
Balanced accuracy: 0.346
Macro F1:          0.280
Per-recipe recall:
  Recipe 5: recall=0.077, support=13
  Recipe 6: recall=0.000, support=12
  Recipe 7: recall=0.333, support=12
  Recipe 8: recall=1.000, support=18
  Recipe 9: recall=0.783, support=23
  Recipe 10: recall=0.000, support=19
  Recipe 11: recall=0.000, support=20
  Recipe 12: recall=0.318, support=22
  Recipe 13: recall=0.947, support=19
  Recipe 14: recall=0.000, support=11

Two-camera artifacts saved in: c:\Users\varelaa\Hackathon 2026\Training model 1\cleaned_data\vial_tank_resnet18_run


# Results and interpretation

The two-camera model reached **39.1% accuracy**, **34.6% balanced accuracy**, and **28.0% macro F1** on the 169 held-out image pairs.

| Model | Accuracy | Balanced accuracy | Macro F1 |
|---|---:|---:|---:|
| Vial only | 43.8% | 39.6% | 34.2% |
| Tank only | 27.2% | 27.3% | 16.2% |
| Vial + tank | 39.1% | 34.6% | 28.0% |

Combining the cameras improved on the tank-only model, but did not outperform the vial-only model on this test split. The tank view therefore did not add enough useful signal in this run to offset the fusion model's added complexity. Recipe-level recall is uneven; several recipes were not recognized in the held-out set. The test contains one held-out fill per recipe from the same recording setup, so these results are a small proof of concept, not evidence of performance on new production runs. Use additional fills and independently recorded sessions for a stronger evaluation.

## Training-versus-test accuracy

To measure the train/test gap, each saved model checkpoint was evaluated in inference mode on both the same 340 training frames (or paired examples) it learned from and the 169 held-out frames. This is a post-training, in-sample accuracy measurement; the training loop recorded loss, not accuracy. The image preprocessing and held-out split are the same as in the model experiments.

| Model | Training accuracy (340 examples) | Test accuracy (169 examples) | Train-test gap |
|---|---:|---:|---:|
| Vial only | 100.0% | 43.8% | 56.2 percentage points |
| Tank only | 40.9% | 27.2% | 13.7 percentage points |
| Vial + tank | 99.4% | 39.1% | 60.4 percentage points |

The vial-only and two-camera models fit their training examples extremely well, but their accuracy drops sharply on the held-out fills. These large gaps are a warning sign of overfitting: the models may have learned visual details specific to the training fills rather than recipe patterns that generalize to a new fill. The tank-only model has a smaller gap, but its training and test accuracies are both low, suggesting it did not learn a strong recipe classifier from these tank images. A smaller gap by itself is not a sign of a better model.

Because nearby frames from each fill are visually similar, the split keeps complete vial fills together; that makes the test set a more meaningful check than a random frame split. Still, it includes only one held-out fill per recipe from this recording. More independently recorded fills are needed before drawing firm conclusions about performance on future production runs.
